# VAECox — Full Reproduction on Real TCGA Data (Kaggle GPU)

**Reproducibility study of** *Kim, Kim, Choe, Lee & Kang (2020),
"Improved survival analysis by learning shared genomic information from
pan-cancer data", Bioinformatics 36(Suppl_1):i389–i398.*
DOI: 10.1093/bioinformatics/btaa462 · Code: https://github.com/dmis-lab/VAECox

This single notebook runs the **entire** pipeline end-to-end:

1. Read real TCGA RNA-seq + survival from the attached **GenoTEX** dataset
   (`input/TCGA/` — sourced from UCSC Xena, open access, no dbGaP).
2. Preprocess (expression is already log2 → per-gene z-normalisation).
3. Pretrain the **VAE** on pan-cancer expression (GPU).
4. Train + evaluate all survival models, including the **fine-tuned VAECox**
   (the paper's actual method — encoder is *unfrozen*).
5. Reproduce the headline claim: **C-index, VAECox vs baselines on 10 cancers**.
6. Run the extensions: robustness, fairness, lightweight models, feature importance, Kaplan–Meier.
7. Write all result CSVs, figures, and a reproducibility card to `/kaggle/working`.

### How to run on Kaggle
* Add data → attach **GenoTEX: LLM Agent Benchmark for Genomic Analysis** (haoyangliu14).
* Settings → Accelerator → **GPU T4 x2** (or P100). Internet can be **Off** (data is local).
* Run all cells. Checkpoints + results land in `/kaggle/working` and persist as notebook output.
* The heavy cells (VAE pretrain, Phase 2) print progress and save intermediate files, so a
  12-hour session timeout never loses completed work — just re-run and it resumes from cache.

> **DATA NOTE (read once):** the loader auto-discovers `input/TCGA/` under `/kaggle/input`,
> extracts the cohort code from each filename (`TCGA.BLCA.sampleMap_...`), and pulls overall
> survival from the `clinicalMatrix`. Check the printed per-cohort event counts — they should be
> in the dozens–hundreds (real cohorts), not single digits (toy data).

## 0 · Config & environment

In [ ]:
# Kaggle's base image lacks lifelines — install it before anything imports it.
try:
    import lifelines  # noqa: F401
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "lifelines"], check=True)
    print("installed lifelines")

In [ ]:
import os, sys, gc, io, gzip, time, json, math, urllib.request, warnings
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F

warnings.filterwarnings("ignore")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"PyTorch {torch.__version__} | device = {DEVICE}")
if DEVICE.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")

# ---- Reproducibility config (mirrors the paper / repo) ----------------------
CFG = dict(
    # 10 cancers evaluated in the paper (Table 1). Whichever of these are present
    # in the attached data get evaluated; ALL loaded cohorts feed VAE pretraining.
    PAPER_10   = ["BLCA", "BRCA", "HNSC", "KIRC", "LGG",
                  "LIHC", "LUAD", "LUSC", "OV", "STAD"],
    HIDDEN     = 4096,     # VAE hidden layer  (paper)
    LATENT     = 128,      # VAE latent dim    (paper)
    VAE_EPOCHS = 500,      # paper: 500  (set to 50 for a fast smoke-test)
    VAE_LR     = 1e-3,
    VAE_WD     = 1e-5,
    VAE_BATCH  = 256,      # minibatch for GPU efficiency (paper used full batch on GPU)
    SURV_EPOCHS= 100,
    SEEDS      = list(range(10)),   # paper: 10 seeds
    HP_SEARCH  = True,     # reduced grid, searched once per cancer (see §5)
    OUT        = "/kaggle/working" if os.path.isdir("/kaggle/working") else "./out",
)
os.makedirs(CFG["OUT"], exist_ok=True)
os.makedirs(f'{CFG["OUT"]}/results', exist_ok=True)
os.makedirs(f'{CFG["OUT"]}/figures', exist_ok=True)


def set_seed(s):
    np.random.seed(s); torch.manual_seed(s)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(s)


print("Cancers to evaluate (if present in data):", CFG["PAPER_10"])

## 1 · Load real TCGA data from the attached GenoTEX dataset

The GenoTEX benchmark ships `input/TCGA/`, downloaded directly from the UCSC
Xena TCGA Hub: one folder per cancer, each with a `HiSeqV2_PANCAN` expression
matrix (already log2, pan-cancer normalised) and a `clinicalMatrix` holding
survival. We auto-discover the folder, read the cohort code from the filename
(`TCGA.BLCA.sampleMap_...`), and extract overall survival (time + event).

No internet needed — everything is mounted read-only under `/kaggle/input`.

In [ ]:
import glob, re

# Auto-discover the TCGA folder (mount path differs from the dataset URL).
_cands = glob.glob("/kaggle/input/**/input/TCGA", recursive=True) or \
         glob.glob("/kaggle/input/**/TCGA", recursive=True) or \
         glob.glob("./**/input/TCGA", recursive=True)
assert _cands, ("GenoTEX TCGA folder not found under /kaggle/input — "
                "attach the GenoTEX dataset (Add data).")
TCGA_ROOT = _cands[0]
print("TCGA_ROOT =", TCGA_ROOT)


def _resolve(path):
    """Kaggle unzips .gz files into a *directory* — descend to the real data file."""
    if os.path.isfile(path):
        return path
    if os.path.isdir(path):
        best, best_sz = None, -1
        for r, _, fs in os.walk(path):
            for fn in fs:
                p = os.path.join(r, fn)
                sz = os.path.getsize(p)
                if sz > best_sz:
                    best, best_sz = p, sz
        return best
    return None


def _read_tsv(path):
    """Read a Xena TSV; detect gzip by magic bytes (filename may lack .gz)."""
    path = _resolve(path)
    with open(path, "rb") as fh:
        gzipped = fh.read(2) == b"\x1f\x8b"
    if gzipped:
        with gzip.open(path, "rt") as f:
            return pd.read_csv(f, sep="\t", index_col=0)
    return pd.read_csv(path, sep="\t", index_col=0)


def parse_survival(clin):
    """Extract overall survival from a Xena clinicalMatrix.
       Returns DataFrame(index=sample) with 'survival' (days) + 'censored' (0=event,1=censored)."""
    C = {c.lower(): c for c in clin.columns}
    # 1) explicit OS time + event indicator (several Xena vintages)
    for tcol, ecol in [("os.time", "os"), ("_os", "_os_ind"),
                       ("_time_to_event", "_event"), ("os_time", "os_status")]:
        if tcol in C and ecol in C:
            t = pd.to_numeric(clin[C[tcol]], errors="coerce")
            e = pd.to_numeric(clin[C[ecol]], errors="coerce")
            df = pd.DataFrame({"survival": t, "censored": (1 - e)}).dropna()
            if len(df) > 10:
                df["censored"] = df["censored"].astype(int)
                return df
    # 2) derive from vital_status + days_to_death / days_to_last_followup
    if "vital_status" in C:
        vs = clin[C["vital_status"]].astype(str).str.upper()
        dead = vs.str.startswith("DEAD") | vs.str.contains("DECEAS")
        dtd = pd.to_numeric(clin[C["days_to_death"]], errors="coerce") \
              if "days_to_death" in C else pd.Series(np.nan, index=clin.index)
        dtf = pd.to_numeric(clin[C["days_to_last_followup"]], errors="coerce") \
              if "days_to_last_followup" in C else pd.Series(np.nan, index=clin.index)
        surv = np.where(dead, dtd, dtf)
        df = pd.DataFrame({"survival": surv, "censored": (~dead).astype(int)},
                          index=clin.index).dropna()
        return df
    return None


def load_genotex_cohort(folder):
    """Return (cohort_code, DataFrame[genes + survival + censored]) or (code, None)."""
    files = os.listdir(folder)
    exp_f = next((f for f in files if "HiSeqV2_PANCAN" in f), None) or \
            next((f for f in files if "HiSeqV2" in f), None)
    cli_f = next((f for f in files if "clinicalMatrix" in f), None)
    if not exp_f or not cli_f:
        return os.path.basename(folder), None
    m = re.search(r"TCGA\.([A-Za-z]+)\.sampleMap", exp_f)
    code = m.group(1).upper() if m else os.path.basename(folder).upper()

    expr = _read_tsv(os.path.join(folder, exp_f)).T          # samples x genes
    expr.index = expr.index.astype(str).str[:15]
    expr = expr[~expr.index.duplicated(keep="first")]

    surv = parse_survival(_read_tsv(os.path.join(folder, cli_f)))
    if surv is None:
        return code, None
    surv.index = surv.index.astype(str).str[:15]
    surv = surv[~surv.index.duplicated(keep="first")]

    common = [s for s in expr.index.intersection(surv.index) if s[13:15] == "01"]
    if len(common) < 20:
        return code, None
    df = expr.loc[common].dropna(axis=1)
    df["survival"] = surv.loc[common, "survival"].astype(float).values
    df["censored"] = surv.loc[common, "censored"].astype(int).values
    return code, df[df["survival"] > 0]


# ---------- Load every cohort folder ----------
COHORT_DFS = {}
for folder in sorted(glob.glob(os.path.join(TCGA_ROOT, "*"))):
    if not os.path.isdir(folder):
        continue
    code, df = load_genotex_cohort(folder)
    if df is not None:
        COHORT_DFS[code] = df
    else:
        print(f"  skip {code} (no usable expression/survival)")

if not COHORT_DFS:
    raise RuntimeError("No usable TCGA cohorts loaded from GenoTEX.")

# Genes shared across all loaded cohorts → consistent VAE input dim.
GENES = sorted(set.intersection(*[set(d.columns) - {"survival", "censored"}
                                  for d in COHORT_DFS.values()]))
NUM_FEATURES = len(GENES)

# Evaluate whichever of the paper's 10 are present; all cohorts feed VAE pretraining.
CFG["PAPER_10"] = [c for c in CFG["PAPER_10"] if c in COHORT_DFS]
DATA_SOURCE = "genotex"

print(f"\nDATA SOURCE = REAL TCGA (GenoTEX / UCSC Xena)")
print(f"Cohorts loaded ({len(COHORT_DFS)}): {sorted(COHORT_DFS)}")
print(f"Evaluating (paper 10 present): {CFG['PAPER_10']}")
print(f"Shared genes (VAE input dim): {NUM_FEATURES}")
for c in sorted(COHORT_DFS):
    d = COHORT_DFS[c]
    n_ev = int((d['censored'] == 0).sum())
    print(f"  {c:6s}: N={len(d):4d}  events={n_ev:4d}  censor%={100*(d['censored']==1).mean():.0f}")

## 2 · Preprocessing & splits

Per-gene **z-normalisation fit on the training set only** (no leakage), and a
stratified 80/20 split by survival-time quintile — matching the repo's Phase 1.

In [ ]:
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.preprocessing import StandardScaler


def cohort_matrix(cohort):
    d = COHORT_DFS[cohort]
    X = d[GENES].values.astype(np.float32)
    y = d["survival"].values.astype(np.float64)
    c = d["censored"].values.astype(np.int32)
    return X, y, c


def make_split(cohort, seed):
    X, y, c = cohort_matrix(cohort)
    # stratify by survival quintile (fallback to event indicator if too few)
    try:
        strata = pd.qcut(y, q=min(5, len(np.unique(y))), labels=False, duplicates="drop")
    except Exception:
        strata = c
    sss = StratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=seed)
    tr, te = next(sss.split(X, strata))
    sc = StandardScaler().fit(X[tr])
    return (sc.transform(X[tr]).astype(np.float32), sc.transform(X[te]).astype(np.float32),
            y[tr], y[te], c[tr], c[te])


def pancancer_matrix():
    """Stacked z-normalised expression across ALL cohorts for VAE pretraining."""
    mats = []
    for c in COHORT_DFS:
        X, _, _ = cohort_matrix(c)
        mats.append(StandardScaler().fit_transform(X).astype(np.float32))
    return np.vstack(mats)


X_PAN = pancancer_matrix()
print(f"Pan-cancer VAE matrix: {X_PAN.shape}  ({X_PAN.nbytes/1e6:.0f} MB)")

## 3 · VAE model (faithful to the repo's `vae_models.VAE`)

Encoder `p → 4096 → (μ,σ) 128`, decoder `128 → 4096 → p`, Tanh activations,
loss = MSE reconstruction + KL divergence.

In [ ]:
class VAE(nn.Module):
    def __init__(self, num_features, hidden=4096, latent=128, dropout=0.0):
        super().__init__()
        self.encode = nn.Sequential(nn.Linear(num_features, hidden), nn.Tanh(), nn.Dropout(dropout))
        self.encode_mu = nn.Sequential(nn.Linear(hidden, latent), nn.Tanh(), nn.Dropout(dropout))
        self.encode_si = nn.Sequential(nn.Linear(hidden, latent), nn.Tanh(), nn.Dropout(dropout))
        self.decode = nn.Sequential(nn.Linear(latent, hidden), nn.Tanh(), nn.Dropout(dropout),
                                    nn.Linear(hidden, num_features))
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_normal_(m.weight)

    def reparam(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        return mu + torch.randn_like(std) * std

    def embed(self, x):
        return self.encode_mu(self.encode(x))

    def forward(self, x):
        h = self.encode(x)
        mu, logvar = self.encode_mu(h), self.encode_si(h)
        recon = self.decode(mu)
        mse = F.mse_loss(recon, x, reduction="mean")
        kld = -0.5 * torch.mean(1 + logvar - mu.pow(2) - logvar.exp())
        return mse + kld


def _find_pretrained():
    """Look for a pretrained VAE: working-dir cache first, then any attached dataset."""
    local = f'{CFG["OUT"]}/vae_pretrained.pt'
    if os.path.exists(local):
        return local
    hits = (glob.glob("/kaggle/input/**/vae_pretrained.pt", recursive=True) or
            glob.glob("/kaggle/input/**/*.pt", recursive=True))
    return hits[0] if hits else None


def train_vae():
    ckpt = _find_pretrained()
    if ckpt:
        print(f"Loading pretrained VAE (skipping training): {ckpt}")
        vae = VAE(NUM_FEATURES, CFG["HIDDEN"], CFG["LATENT"]).to(DEVICE)
        try:
            vae.load_state_dict(torch.load(ckpt, map_location=DEVICE))
        except RuntimeError as e:
            raise RuntimeError(
                f"Checkpoint shape mismatch — it was trained with a different "
                f"gene set than the current NUM_FEATURES={NUM_FEATURES}. "
                f"Delete the attached .pt to retrain, or re-use the exact data. "
                f"Original error: {e}")
        return vae
    set_seed(0)
    vae = VAE(NUM_FEATURES, CFG["HIDDEN"], CFG["LATENT"]).to(DEVICE)
    opt = torch.optim.Adam(vae.parameters(), lr=CFG["VAE_LR"], weight_decay=CFG["VAE_WD"])
    X = torch.tensor(X_PAN, dtype=torch.float32)
    n, bs = X.shape[0], CFG["VAE_BATCH"]
    print(f"Training VAE: {CFG['VAE_EPOCHS']} epochs, {n} samples, batch {bs}")
    t0 = time.time()
    for ep in range(CFG["VAE_EPOCHS"]):
        vae.train(); perm = torch.randperm(n); tot = 0.0
        for i in range(0, n, bs):
            xb = X[perm[i:i+bs]].to(DEVICE)
            opt.zero_grad(); loss = vae(xb); loss.backward(); opt.step()
            tot += loss.item() * len(xb)
        if ep % 25 == 0 or ep == CFG["VAE_EPOCHS"] - 1:
            print(f"  epoch {ep:3d}  loss {tot/n:.4f}  ({time.time()-t0:.0f}s)")
    torch.save(vae.state_dict(), ckpt)
    print(f"VAE trained in {time.time()-t0:.0f}s → {ckpt}")
    return vae


VAE_MODEL = train_vae()

## 4 · Survival models & Cox partial-likelihood loss

`PartialNLL`, risk-set matrix and C-index are ported directly from the repo
(`models.py`, `phase2_reproduction.py`). VAECox here is the **fine-tuned**
variant: pretrained encoder + Coxnnet head, all weights trainable.

In [ ]:
import copy
from lifelines.utils import concordance_index


class PartialNLL(nn.Module):
    def forward(self, theta, R, censored):
        observed = 1 - censored
        num_obs = torch.sum(observed)
        if num_obs == 0:
            return (theta * 0).sum()
        exp_theta = torch.exp(theta)
        return -(torch.sum((theta.reshape(-1) -
                 torch.log(torch.sum(exp_theta * R.t(), 0))) * observed) / num_obs)


class CoxLinear(nn.Module):
    def __init__(self, p):
        super().__init__(); self.fc1 = nn.Linear(p, 1); nn.init.xavier_normal_(self.fc1.weight)
    def forward(self, x): return self.fc1(x)


class Coxnnet(nn.Module):
    def __init__(self, p):
        super().__init__(); h = int(np.ceil(p ** 0.5))
        self.fc1 = nn.Linear(p, h); self.fc2 = nn.Linear(h, 1)
    def forward(self, x): return self.fc2(torch.tanh(self.fc1(x)))


class CoxMLP(nn.Module):
    def __init__(self, p, nhid=100, dropout=0.0):
        super().__init__(); self.fc1 = nn.Linear(p, nhid); self.fc2 = nn.Linear(nhid, 1); self.d = dropout
    def forward(self, x):
        x = F.dropout(F.relu(self.fc1(x)), self.d, training=self.training); return self.fc2(x)


class VAECox(nn.Module):
    """Paper's method: pretrained VAE encoder (FINE-TUNED) + Coxnnet(128).

    The encoder is deep-copied. Assigning `pretrained_vae.encode` directly would
    share the module object, so every fine-tuning run would keep mutating the one
    pretrained VAE in place and each cancer/seed would silently start from the
    previous run's weights.
    """
    def __init__(self, pretrained_vae, latent=128):
        super().__init__()
        vae = copy.deepcopy(pretrained_vae)
        self.encode = vae.encode
        self.encode_mu = vae.encode_mu
        self.cox = Coxnnet(latent)
        for p in self.parameters():
            p.requires_grad = True
    def forward(self, x):
        return self.cox(self.encode_mu(self.encode(x)))


def make_R(y):
    n = len(y); R = np.zeros((n, n), dtype=np.float32)
    for i in range(n): R[i, :] = (y >= y[i])
    return R


def cindex_safe(y, pred, c):
    ev = (c == 0)
    if ev.sum() == 0: return float("nan")
    try: return concordance_index(y, pred, ev)
    except Exception: return float("nan")


def train_eval(model, Xtr, ytr, ctr, Xte, yte, cte, lr, wd, epochs, lasso=0.0):
    model = model.to(DEVICE)
    lossf = PartialNLL()
    X = torch.tensor(Xtr, dtype=torch.float32, device=DEVICE)
    R = torch.tensor(make_R(ytr), dtype=torch.float32, device=DEVICE)
    c = torch.tensor(ctr, dtype=torch.float32, device=DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=wd)
    model.train()
    for _ in range(epochs):
        opt.zero_grad(); theta = model(X); loss = lossf(theta, R, c)
        if lasso > 0:
            loss = loss + lasso * sum(p.abs().sum() for p in model.fc1.parameters())
        if torch.isnan(loss) or torch.isinf(loss): break
        loss.backward(); opt.step()
    model.eval()
    with torch.no_grad():
        pred = -model(torch.tensor(Xte, dtype=torch.float32, device=DEVICE)).reshape(-1).cpu().numpy()
    return cindex_safe(yte, pred, cte)

## 5 · Phase 2 — reproduce the headline C-index table

All models on all 10 cancers × 10 seeds, with a **reduced hyperparameter
search** (lr × weight-decay) done once per cancer on a validation split, then
applied across seeds. Reports mean ± std → `results/cindex_comparison.csv`.

In [ ]:
HP_GRID = [(1e-3, 1e-5), (1e-3, 1e-3), (1e-4, 1e-5)] if CFG["HP_SEARCH"] else [(1e-3, 1e-5)]


def build_model(name, p):
    if name == "CoxLasso":  return CoxLinear(p)
    if name == "CoxRidge":  return CoxLinear(p)
    if name == "Coxnnet":   return Coxnnet(p)
    if name == "CoxMLP":    return CoxMLP(p)
    if name == "VAECox":    return VAECox(VAE_MODEL, CFG["LATENT"])
    raise ValueError(name)


def fit_one(name, Xtr, ytr, ctr, Xte, yte, cte, lr, wd):
    lasso = 0.01 if name == "CoxLasso" else 0.0
    wd = 1e-3 if name == "CoxRidge" else wd
    m = build_model(name, Xtr.shape[1])
    return train_eval(m, Xtr, ytr, ctr, Xte, yte, cte, lr, wd, CFG["SURV_EPOCHS"], lasso)


def search_hp(name, cohort):
    """Pick (lr,wd) on seed-0 val split; return best-scoring combo."""
    if len(HP_GRID) == 1: return HP_GRID[0]
    Xtr, Xte, ytr, yte, ctr, cte = make_split(cohort, 0)
    best, best_hp = -1, HP_GRID[0]
    for lr, wd in HP_GRID:
        ci = fit_one(name, Xtr, ytr, ctr, Xte, yte, cte, lr, wd)
        if not np.isnan(ci) and ci > best: best, best_hp = ci, (lr, wd)
    return best_hp


MODELS = ["CoxLasso", "CoxRidge", "Coxnnet", "CoxMLP", "VAECox"]


def run_phase2():
    rows = []
    for cohort in CFG["PAPER_10"]:
        if cohort not in COHORT_DFS:
            continue
        print(f"\n── {cohort} ──")
        hp = {m: search_hp(m, cohort) for m in MODELS}
        for m in MODELS:
            vals = []
            for seed in CFG["SEEDS"]:
                set_seed(seed)
                Xtr, Xte, ytr, yte, ctr, cte = make_split(cohort, seed)
                lr, wd = hp[m]
                vals.append(fit_one(m, Xtr, ytr, ctr, Xte, yte, cte, lr, wd))
            v = [x for x in vals if not np.isnan(x)]
            mean = np.mean(v) if v else float("nan")
            std  = np.std(v) if v else float("nan")
            rows.append(dict(cancer=cohort, model=m, mean_cindex=round(mean, 4),
                             std_cindex=round(std, 4), n_valid=len(v)))
            print(f"  {m:9s}: {mean:.3f} ± {std:.3f}  (hp={hp[m]}, {len(v)} seeds)")
    df = pd.DataFrame(rows)
    df.to_csv(f'{CFG["OUT"]}/results/cindex_long.csv', index=False)
    # wide table (mean) + wins
    wide = df.pivot(index="model", columns="cancer", values="mean_cindex")
    wide["Mean"] = wide.mean(axis=1)
    wide.to_csv(f'{CFG["OUT"]}/results/cindex_comparison.csv')
    wins = {m: 0 for m in MODELS}
    for cohort in wide.columns[:-1]:
        col = wide[cohort].dropna()
        if len(col): wins[col.idxmax()] += 1
    print("\n=== WINS (VAECox target: 7/10) ===")
    for m, w in sorted(wins.items(), key=lambda x: -x[1]):
        print(f"  {m:9s}: {w}/10")
    return df, wide, wins


PH2_LONG, PH2_WIDE, PH2_WINS = run_phase2()
print("\n", PH2_WIDE.round(3))

## 6 · Phase 3 — extensions

Robustness (missing + noise), fairness (event-rate vs C-index), lightweight
models (latent/hidden sweep), feature importance, Kaplan–Meier. All on the
real data, comparing standard Cox (Ridge) vs VAECox.

In [ ]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from lifelines import KaplanMeierFitter
from lifelines.statistics import logrank_test

RES = f'{CFG["OUT"]}/results'


# --- 6a robustness: missing features + gaussian noise -----------------------
def robustness(cohort="STAD"):
    out = []
    for frac in [0.0, 0.1, 0.25, 0.5]:
        cis = {"CoxRidge": [], "VAECox": []}
        for seed in CFG["SEEDS"]:
            set_seed(seed)
            Xtr, Xte, ytr, yte, ctr, cte = make_split(cohort, seed)
            rng = np.random.default_rng(seed + 1000)
            mask = rng.random(Xte.shape) >= frac
            Xte_m = Xte * mask
            cis["CoxRidge"].append(fit_one("CoxRidge", Xtr, ytr, ctr, Xte_m, yte, cte, 1e-4, 1e-3))
            cis["VAECox"].append(fit_one("VAECox", Xtr, ytr, ctr, Xte_m, yte, cte, 1e-3, 1e-5))
        out.append(dict(experiment="missing", level=f"{int(frac*100)}%",
                        CoxRidge=np.nanmean(cis["CoxRidge"]), VAECox=np.nanmean(cis["VAECox"])))
    for sig in [0.0, 0.5, 1.0, 2.0]:
        cis = {"CoxRidge": [], "VAECox": []}
        for seed in CFG["SEEDS"]:
            set_seed(seed)
            Xtr, Xte, ytr, yte, ctr, cte = make_split(cohort, seed)
            rng = np.random.default_rng(seed + 2000)
            Xte_n = Xte + rng.normal(0, sig, Xte.shape).astype(np.float32)
            cis["CoxRidge"].append(fit_one("CoxRidge", Xtr, ytr, ctr, Xte_n, yte, cte, 1e-4, 1e-3))
            cis["VAECox"].append(fit_one("VAECox", Xtr, ytr, ctr, Xte_n, yte, cte, 1e-3, 1e-5))
        out.append(dict(experiment="noise", level=f"sigma={sig}",
                        CoxRidge=np.nanmean(cis["CoxRidge"]), VAECox=np.nanmean(cis["VAECox"])))
    df = pd.DataFrame(out); df.to_csv(f"{RES}/robustness.csv", index=False)
    print(df.round(3)); return df


# --- 6b fairness: does C-index track #events / cohort size? ------------------
def fairness():
    rows = []
    for cohort in CFG["PAPER_10"]:
        if cohort not in COHORT_DFS: continue
        sub = PH2_LONG[(PH2_LONG.cancer == cohort) & (PH2_LONG.model == "VAECox")]
        if len(sub) == 0: continue
        d = COHORT_DFS[cohort]
        rows.append(dict(cancer=cohort, n=len(d), n_events=int((d.censored == 0).sum()),
                         vaecox_cindex=float(sub.mean_cindex.iloc[0])))
    df = pd.DataFrame(rows); df.to_csv(f"{RES}/fairness.csv", index=False)
    if len(df) > 2:
        r = np.corrcoef(df.n_events, df.vaecox_cindex)[0, 1]
        print(f"corr(events, VAECox C-index) = {r:.3f}")
    return df


# --- 6c lightweight: latent/hidden dimension sweep --------------------------
def lightweight(cohort="STAD"):
    rows = []
    for hidden, latent in [(512, 128), (1024, 128), (4096, 32), (4096, 64), (4096, 128)]:
        set_seed(0)
        vae = VAE(NUM_FEATURES, hidden, latent).to(DEVICE)
        opt = torch.optim.Adam(vae.parameters(), lr=CFG["VAE_LR"], weight_decay=CFG["VAE_WD"])
        X = torch.tensor(X_PAN, dtype=torch.float32); n, bs = X.shape[0], CFG["VAE_BATCH"]
        t0 = time.time()
        for ep in range(min(100, CFG["VAE_EPOCHS"])):
            perm = torch.randperm(n)
            for i in range(0, n, bs):
                xb = X[perm[i:i+bs]].to(DEVICE)
                opt.zero_grad(); vae(xb).backward(); opt.step()
        train_sec = time.time() - t0
        cis = []
        for seed in CFG["SEEDS"]:
            set_seed(seed)
            Xtr, Xte, ytr, yte, ctr, cte = make_split(cohort, seed)
            m = VAECox(vae, latent)
            cis.append(train_eval(m, Xtr, ytr, ctr, Xte, yte, cte, 1e-3, 1e-5, CFG["SURV_EPOCHS"]))
        n_params = sum(p.numel() for p in vae.parameters())
        rows.append(dict(hidden=hidden, latent=latent, n_params=n_params,
                         train_sec=round(train_sec, 1), mean_cindex=round(np.nanmean(cis), 4)))
        print(rows[-1])
    df = pd.DataFrame(rows); df.to_csv(f"{RES}/lightweight.csv", index=False); return df


# --- 6d feature importance: |Cox weights| on Ridge --------------------------
def feature_importance(cohort="STAD", topk=25):
    set_seed(0)
    Xtr, Xte, ytr, yte, ctr, cte = make_split(cohort, 0)
    m = CoxLinear(Xtr.shape[1]).to(DEVICE)
    train_eval(m, Xtr, ytr, ctr, Xte, yte, cte, 1e-4, 1e-3, CFG["SURV_EPOCHS"])
    w = m.fc1.weight.detach().cpu().numpy().reshape(-1)
    idx = np.argsort(-np.abs(w))[:topk]
    df = pd.DataFrame(dict(gene=[GENES[i] for i in idx], weight=w[idx].round(4)))
    df.to_csv(f"{RES}/feature_importance.csv", index=False)
    print(df.head(10)); return df


# --- 6e Kaplan-Meier by predicted risk --------------------------------------
def kaplan_meier(cohort="STAD"):
    set_seed(0)
    Xtr, Xte, ytr, yte, ctr, cte = make_split(cohort, 0)
    m = VAECox(VAE_MODEL, CFG["LATENT"]).to(DEVICE)
    train_eval(m, Xtr, ytr, ctr, Xte, yte, cte, 1e-3, 1e-5, CFG["SURV_EPOCHS"])
    m.eval()
    with torch.no_grad():
        risk = m(torch.tensor(Xte, dtype=torch.float32, device=DEVICE)).reshape(-1).cpu().numpy()
    hi = risk >= np.median(risk)
    ev = (cte == 0)
    fig, ax = plt.subplots(figsize=(6, 4))
    kmf = KaplanMeierFitter()
    for grp, lab in [(hi, "High risk"), (~hi, "Low risk")]:
        if grp.sum() > 0:
            kmf.fit(yte[grp], ev[grp], label=lab); kmf.plot_survival_function(ax=ax)
    lr = logrank_test(yte[hi], yte[~hi], ev[hi], ev[~hi])
    ax.set_title(f"{cohort} — KM by VAECox risk (log-rank p={lr.p_value:.3f})")
    ax.set_xlabel("Days"); ax.set_ylabel("Survival probability")
    fig.tight_layout(); fig.savefig(f'{CFG["OUT"]}/figures/km_{cohort}.png', dpi=120)
    print(f"{cohort} log-rank p = {lr.p_value:.4f}")
    return lr.p_value


print("\n### 6a robustness"); ROB = robustness()
print("\n### 6b fairness");   FAIR = fairness()
print("\n### 6c lightweight");LIGHT = lightweight()
print("\n### 6d importance"); FI = feature_importance()
print("\n### 6e Kaplan-Meier")
for c in ["STAD", "BLCA", "KIRC"]:
    if c in COHORT_DFS: kaplan_meier(c)

## 6g · Extension helpers + clinical metadata

`train_eval` only ever returns a C-index, but the fairness, robustness,
interpretability and Kaplan–Meier extensions all need the **risk scores**
themselves and the identity of the test patients. This cell adds a reusable
fit/predict pair, an index-returning version of the split, and re-reads the
clinical columns (`age`, `gender`, `stage`, `histological_type`) that §1 dropped
when it kept only survival.

In [ ]:
# Bar chart of mean C-index per model
fig, ax = plt.subplots(figsize=(7, 4))
PH2_WIDE["Mean"].sort_values().plot.barh(ax=ax, color="#4C78A8")
ax.set_xlabel("Mean C-index"); ax.set_title("VAECox reproduction — mean C-index across 10 cancers")
fig.tight_layout(); fig.savefig(f'{CFG["OUT"]}/figures/mean_cindex.png', dpi=120)

# Robustness figure
fig, ax = plt.subplots(figsize=(7, 4))
miss = ROB[ROB.experiment == "missing"]
ax.plot(range(len(miss)), miss.CoxRidge, "-o", label="CoxRidge")
ax.plot(range(len(miss)), miss.VAECox, "-o", label="VAECox")
ax.set_xticks(range(len(miss))); ax.set_xticklabels(miss.level)
ax.set_xlabel("Missing features"); ax.set_ylabel("C-index"); ax.legend()
ax.set_title("Robustness to missing features")
fig.tight_layout(); fig.savefig(f'{CFG["OUT"]}/figures/robustness.png', dpi=120)


# ---- extension summary lines, computed rather than typed --------------------
def _fmt_wins():
    order = sorted(PH2_WINS.items(), key=lambda x: -x[1])
    return ", ".join(f"{m} {w}" for m, w in order)


_gap_line = "not computed"
if len(GAPS):
    worst = GAPS.iloc[0]
    _gap_line = (f"largest gap {worst.gap:.3f} C-index ({worst.cancer}/{worst.variable}, "
                 f"{worst.model}: {worst.best_group} vs {worst.worst_group}); "
                 f"mean gap across strata {GAPS.gap.mean():.3f}")

_light_line = "not computed"
if len(LIGHT_GAP):
    sm = LIGHT_GAP[LIGHT_GAP.n_events <= LIGHT_GAP.n_events.median()].delta_vs_full.mean()
    lg = LIGHT_GAP[LIGHT_GAP.n_events > LIGHT_GAP.n_events.median()].delta_vs_full.mean()
    _light_line = (f"shrinking the VAE changes C-index by {sm:+.3f} on low-event cohorts "
                   f"vs {lg:+.3f} on high-event cohorts")

_rob_line = "not computed"
if len(ROB_ALL):
    for exp, lev in [("missing", "50%"), ("noise", "sigma=2.0")]:
        s = ROB_ALL[(ROB_ALL.experiment == exp) & (ROB_ALL.level == lev)]
        c0 = ROB_ALL[(ROB_ALL.experiment == exp)].groupby("level").mean(numeric_only=True).iloc[0]
        if len(s):
            _rob_line = (f"at {exp}={lev}: CoxRidge {s.CoxRidge.mean():.3f} "
                         f"(from {c0.CoxRidge:.3f}), VAECox {s.VAECox.mean():.3f} "
                         f"(from {c0.VAECox:.3f})")

_km_line = ("not computed" if not len(KM_ALL) else
            f"{int(KM_ALL.significant.fillna(False).sum())}/{len(KM_ALL)} cohorts show "
            f"significant high/low risk separation (log-rank p<0.05)")

_cpu_line = "not computed"
if len(FEATSUB):
    cpu_rows = FEATSUB[FEATSUB.device == "cpu"]
    if len(cpu_rows):
        r = cpu_rows.iloc[0]
        _cpu_line = (f"k={int(r.k_genes)} genes, {r.vae_params/1e6:.2f}M-param VAE trains on CPU in "
                     f"{r.vae_pretrain_sec}s; C-index {r.mean_cindex}")

card = f"""
================================================================================
REPRODUCIBILITY CARD — VAECox (Bioinformatics 2020, Suppl. 1)
================================================================================
Paper : Kim, Kim, Choe, Lee, Kang. "Improved survival analysis by learning
        shared genomic information from pan-cancer data."
        Bioinformatics 36(Suppl_1):i389-i398. DOI:10.1093/bioinformatics/btaa462
Claim : VAECox outperforms CoxLasso/CoxRidge/Coxnnet on 7/10 TCGA cancers (C-index).

DATA SOURCE     : REAL TCGA via GenoTEX (UCSC Xena TCGA Hub, HiSeqV2_PANCAN)
Cohorts         : {sorted(COHORT_DFS)}
Genes (VAE dim) : {NUM_FEATURES}
Device          : {DEVICE} ({torch.cuda.get_device_name(0) if DEVICE.type=='cuda' else 'CPU'})
Seeds           : {CFG['SEEDS']}

VAE             : {NUM_FEATURES}->{CFG['HIDDEN']}->{CFG['LATENT']} (mu,sigma), Tanh, Adam
                  lr={CFG['VAE_LR']} wd={CFG['VAE_WD']} epochs={CFG['VAE_EPOCHS']} batch={CFG['VAE_BATCH']}
VAECox          : pretrained encoder FINE-TUNED + Coxnnet(128)  [paper's method]
HP search       : {'reduced grid ' + str(HP_GRID) + ' per cancer' if CFG['HP_SEARCH'] else 'none'}
Surv epochs     : {CFG['SURV_EPOCHS']}

--------------------------------------------------------------------------------
PHASE 2 — REPRODUCTION
--------------------------------------------------------------------------------
Wins (this run) : {_fmt_wins()}
Paper wins      : VAECox 7/10
Mean C-index    : {', '.join(f'{m} {v:.3f}' for m, v in PH2_WIDE['Mean'].items())}
Best mean       : {PH2_WIDE['Mean'].idxmax()}

--------------------------------------------------------------------------------
PHASE 3 — EXTENSIONS
--------------------------------------------------------------------------------
Subgroup fairness   : {_gap_line}
Lightweight equity  : {_light_line}
Robustness          : {_rob_line}
Risk stratification : {_km_line}
CPU accessibility   : {_cpu_line}
Interpretability    : permutation importance over {PERM_IMP.cancer.nunique() if len(PERM_IMP) else 0} cohorts
                      x {PERM_IMP.model.nunique() if len(PERM_IMP) else 0} models

--------------------------------------------------------------------------------
DEVIATIONS
--------------------------------------------------------------------------------
  - HP search reduced vs paper's 18-combo 5-fold CV (time). Documented.
  - Extensions use fixed HP ({DEFAULT_HP}) rather than the per-cancer search,
    so extension C-indices are not directly comparable to the Phase 2 table.
  - VAE minibatched on GPU (paper full-batch); numerically equivalent.
  - Lightweight sweep pretrains every config for {SWEEP_VAE_EPOCHS} epochs (not
    {CFG['VAE_EPOCHS']}) so configs are compared at an equal compute budget.
  - Robustness/fairness use {len(CFG['SEEDS'][:5])} seeds instead of {len(CFG['SEEDS'])}.
  - GenoTEX HiSeqV2_PANCAN gene set ({NUM_FEATURES}) may differ slightly from
    paper's 20,502 (pan-cancer-normalised vs per-cohort). Documented.
  - Subgroup strata below {MIN_GROUP_N} patients or {MIN_GROUP_EV} events are dropped as
    unestimable rather than reported.
================================================================================
"""
with open(f'{CFG["OUT"]}/results/reproducibility_card.txt', "w") as f:
    f.write(card)
print(card)

print("\nAll outputs written to:", CFG["OUT"])
for root, _, files in os.walk(CFG["OUT"]):
    for fn in files:
        print("  ", os.path.join(root, fn))

## 8 · Next step: the manuscript

This notebook produces every result and figure for Phases 2 and 3. Phase 4 is
assembled from them with no number retyped by hand:

1. Download the whole `/kaggle/working` folder (Output → Download all).
2. Drop it into the repo as `out/` (so you have `out/results/*.csv`,
   `out/figures/*.png`, `out/results/manuscript_numbers.json`).
3. Run `python paper/build_manuscript.py` — it reads `manuscript_numbers.json`
   and writes `paper/manuscript_filled.md` with every table and headline number
   populated from the actual run.
4. Write the prose around those tables in `paper/manuscript.md`; re-run the
   build script whenever results change.

**Before step 3**, transcribe the paper's Table 1 into `paper/paper_table1.csv`
(the template is already there with the right cancer/model rows). §6j.4 uses it
for the per-cell comparison; it is deliberately not hard-coded so that no number
attributed to the original authors is ever guessed.

In [ ]:
# ---------------------------------------------------------------------------
# 6h · Subgroup fairness: C-index within age / sex / stage / subtype strata
# ---------------------------------------------------------------------------
MIN_GROUP_N = 10      # patients in a test-set stratum
MIN_GROUP_EV = 3      # uncensored events in that stratum (C-index is undefined below this)


def subgroup_cindex(models=("CoxRidge", "VAECox"), cohorts=None, seeds=None):
    """Per-seed C-index computed *within* each clinical stratum of the test set.

    Risk scores are only comparable inside one fitted model, so the C-index is
    computed per seed and then averaged — pooling raw scores across seeds would
    mix incomparable scales.
    """
    cohorts = cohorts or CFG["PAPER_10"]
    seeds = seeds if seeds is not None else CFG["SEEDS"]
    rows = []
    for cohort in cohorts:
        sub = CLIN_SUB.get(cohort)
        if sub is None or not len(sub.columns):
            continue
        for name in models:
            acc = {}
            for seed in seeds:
                set_seed(seed)
                tr, te = split_indices(cohort, seed)
                Xtr, Xte, ytr, yte, ctr, cte = split_arrays(cohort, tr, te)
                _, risk = fit_risk(name, Xtr, ytr, ctr, Xte)
                lab = sub.iloc[te]
                for var in SUBGROUP_VARS:
                    if var not in lab.columns:
                        continue
                    vals = lab[var]
                    for lev in pd.unique(vals.dropna()):
                        mask = (vals == lev).values
                        if mask.sum() < MIN_GROUP_N or (cte[mask] == 0).sum() < MIN_GROUP_EV:
                            continue
                        ci = cindex_safe(yte[mask], risk[mask], cte[mask])
                        if not np.isnan(ci):
                            acc.setdefault((var, str(lev)), []).append((ci, int(mask.sum())))
            for (var, lev), v in acc.items():
                cis = [x[0] for x in v]
                rows.append(dict(cancer=cohort, model=name, variable=var, group=lev,
                                 n_seeds=len(cis), mean_n=int(np.mean([x[1] for x in v])),
                                 mean_cindex=round(float(np.mean(cis)), 4),
                                 std_cindex=round(float(np.std(cis)), 4)))
        print(f"  {cohort} done ({sum(r['cancer'] == cohort for r in rows)} strata)")
    df = pd.DataFrame(rows)
    df.to_csv(f"{RES}/subgroup_cindex.csv", index=False)
    return df


def subgroup_gaps(df):
    """Best-group minus worst-group C-index — the disparity actually worth reporting."""
    if not len(df):
        return pd.DataFrame()
    rows = []
    for (cancer, model, var), g in df.groupby(["cancer", "model", "variable"]):
        if len(g) < 2:
            continue
        hi, lo = g.loc[g.mean_cindex.idxmax()], g.loc[g.mean_cindex.idxmin()]
        rows.append(dict(cancer=cancer, model=model, variable=var,
                         best_group=hi.group, best_cindex=hi.mean_cindex,
                         worst_group=lo.group, worst_cindex=lo.mean_cindex,
                         gap=round(float(hi.mean_cindex - lo.mean_cindex), 4),
                         n_groups=len(g)))
    out = pd.DataFrame(rows).sort_values("gap", ascending=False)
    out.to_csv(f"{RES}/subgroup_gaps.csv", index=False)
    if len(out):
        print("\n  Largest disparities:")
        print(out.head(10).to_string(index=False))
        print("\n  Mean gap by variable:")
        print(out.groupby(["variable", "model"]).gap.mean().round(4).to_string())
    return out


# --- cohort-level fairness, both models (§6b only did VAECox) ---------------
def cohort_fairness():
    rows = []
    for cohort in CFG["PAPER_10"]:
        d = COHORT_DFS[cohort]
        for _, r in PH2_LONG[PH2_LONG.cancer == cohort].iterrows():
            rows.append(dict(cancer=cohort, model=r.model, n_patients=len(d),
                             n_events=int((d.censored == 0).sum()),
                             event_rate=round(float((d.censored == 0).mean()), 4),
                             mean_cindex=r.mean_cindex))
    df = pd.DataFrame(rows)
    df.to_csv(f"{RES}/cohort_fairness.csv", index=False)
    print("\n  corr(#events, C-index) per model:")
    for m, g in df.groupby("model"):
        g = g.dropna(subset=["mean_cindex"])
        if len(g) > 2:
            print(f"    {m:9s} events r={np.corrcoef(g.n_events, g.mean_cindex)[0,1]:+.3f}   "
                  f"cohort-size r={np.corrcoef(g.n_patients, g.mean_cindex)[0,1]:+.3f}")
    return df


print("### 6h.1 clinical subgroups"); SUBGROUP = subgroup_cindex()
GAPS = subgroup_gaps(SUBGROUP)
print("\n### 6h.2 cohort-level fairness"); COHORT_FAIR = cohort_fairness()

## 6i · Low-resource extensions

§6c swept VAE sizes on one cohort and reported a single mean. The roadmap asks
two sharper questions:

* **Equity of compression** — if a smaller VAE loses accuracy, does it lose *more*
  on the cohorts that already have the fewest uncensored events? A cheap model
  that only stays accurate on large, well-studied cancers is not accessible.
* **Feature budget & CPU feasibility** — how few genes can be kept before the
  C-index collapses, and can the whole pipeline actually run on a CPU? Gene
  selection here is by pan-cancer **variance only**, which never looks at survival
  labels, so the train/test split stays clean.

In [ ]:
# ---------------------------------------------------------------------------
# 6i · Low-resource extensions
#      (a) does shrinking the VAE hurt small cohorts more than large ones?
#      (b) how few genes can you keep and still predict — and does it fit on CPU?
# ---------------------------------------------------------------------------
LIGHT_CONFIGS = [("h4096_l128", 4096, 128),   # paper size (reference point)
                 ("h1024_l64",  1024,  64),
                 ("h512_l32",    512,  32),
                 ("h256_l16",    256,  16)]
SWEEP_VAE_EPOCHS = min(100, CFG["VAE_EPOCHS"])   # equal budget for every config


def train_vae_dims(hidden, latent, cols=None, epochs=None, device=None, seed=0):
    """Pretrain a VAE of the given size on pan-cancer expression (optionally on a
    gene subset). Returns (vae, seconds, n_params)."""
    device = device or DEVICE
    epochs = epochs or SWEEP_VAE_EPOCHS
    Xp = X_PAN if cols is None else X_PAN[:, cols]
    set_seed(seed)
    vae = VAE(Xp.shape[1], hidden, latent).to(device)
    opt = torch.optim.Adam(vae.parameters(), lr=CFG["VAE_LR"], weight_decay=CFG["VAE_WD"])
    X = torch.tensor(Xp, dtype=torch.float32)
    n, bs = X.shape[0], CFG["VAE_BATCH"]
    t0 = time.time()
    for _ in range(epochs):
        vae.train()
        perm = torch.randperm(n)
        for i in range(0, n, bs):
            xb = X[perm[i:i + bs]].to(device)
            opt.zero_grad(); vae(xb).backward(); opt.step()
    vae.eval()
    return vae, round(time.time() - t0, 1), sum(p.numel() for p in vae.parameters())


# --- 6i.a lightweight models, evaluated on EVERY cohort ---------------------
def lightweight_by_cancer(cohorts=None, seeds=None):
    cohorts = cohorts or CFG["PAPER_10"]
    seeds = seeds if seeds is not None else CFG["SEEDS"][:5]
    rows = []
    for cfg_name, hidden, latent in LIGHT_CONFIGS:
        vae, secs, n_params = train_vae_dims(hidden, latent)
        print(f"  {cfg_name}: {n_params/1e6:.1f}M params, pretrained in {secs}s")
        for cohort in cohorts:
            cis = []
            for seed in seeds:
                set_seed(seed)
                tr, te = split_indices(cohort, seed)
                Xtr, Xte, ytr, yte, ctr, cte = split_arrays(cohort, tr, te)
                _, risk = fit_risk("VAECox", Xtr, ytr, ctr, Xte, vae=vae, latent=latent)
                cis.append(cindex_safe(yte, risk, cte))
            rows.append(dict(config=cfg_name, hidden=hidden, latent=latent,
                             n_params=n_params, train_sec=secs, cancer=cohort,
                             cindex=round(float(np.nanmean(cis)), 4),
                             n_events=int((COHORT_DFS[cohort].censored == 0).sum()),
                             n_patients=int(len(COHORT_DFS[cohort]))))
        del vae; gc.collect()
        if DEVICE.type == "cuda": torch.cuda.empty_cache()
    df = pd.DataFrame(rows)
    df.to_csv(f"{RES}/lightweight_by_cancer.csv", index=False)
    print("\n  mean C-index per config:")
    print(df.groupby("config").cindex.mean().round(4).to_string())
    return df


def lightweight_disparity(light_df):
    """Does compressing the model cost *more* C-index on small / low-event
    cohorts? That is the equity question the roadmap asks: a cheap model that is
    only cheap for well-resourced cancers is not actually accessible."""
    full = LIGHT_CONFIGS[0][0]
    base = light_df[light_df.config == full].set_index("cancer").cindex
    sub = light_df[light_df.config != full].copy()
    sub["delta_vs_full"] = (sub.cindex - sub.cancer.map(base)).round(4)
    sub.to_csv(f"{RES}/lightweight_disparity.csv", index=False)
    med = light_df.groupby("cancer").n_events.first().median()
    print(f"\n  median events across cohorts = {med:.0f}; splitting there:")
    for cfg_name, g in sub.groupby("config"):
        small = g[g.n_events <= med].delta_vs_full.mean()
        large = g[g.n_events > med].delta_vs_full.mean()
        r = (np.corrcoef(g.n_events, g.delta_vs_full)[0, 1]
             if g.delta_vs_full.notna().sum() > 2 else float("nan"))
        print(f"    {cfg_name:11s} Δ small-cohort {small:+.4f} | "
              f"Δ large-cohort {large:+.4f} | corr(events, Δ) = {r:+.3f}")
    return sub


# --- 6i.b feature budget + CPU feasibility ---------------------------------
def top_variance_genes(k):
    """Unsupervised (survival labels never touched) → no leakage into the split."""
    return np.argsort(-X_PAN.var(axis=0))[:k]


def feature_subset_accessibility(cohorts=None, seeds=None,
                                 k_list=(100, 500, 1000, 5000, None),
                                 cpu_check_k=1000):
    cohorts = cohorts or CFG["PAPER_10"][:3]
    seeds = seeds if seeds is not None else CFG["SEEDS"][:5]
    rows = []
    for k in k_list:
        cols = None if k is None else top_variance_genes(k)
        kk = NUM_FEATURES if k is None else k
        vae, vae_secs, n_params = train_vae_dims(512, 32, cols=cols)
        per_model = {"CoxRidge": [], "VAECox": []}
        t0 = time.time()
        for cohort in cohorts:
            for seed in seeds:
                set_seed(seed)
                tr, te = split_indices(cohort, seed)
                Xtr, Xte, ytr, yte, ctr, cte = split_arrays(cohort, tr, te, cols=cols)
                _, r_ridge = fit_risk("CoxRidge", Xtr, ytr, ctr, Xte)
                _, r_vae = fit_risk("VAECox", Xtr, ytr, ctr, Xte, vae=vae, latent=32)
                per_model["CoxRidge"].append(cindex_safe(yte, r_ridge, cte))
                per_model["VAECox"].append(cindex_safe(yte, r_vae, cte))
        surv_secs = round(time.time() - t0, 1)
        for name, v in per_model.items():
            rows.append(dict(k_genes=kk, model=name,
                             mean_cindex=round(float(np.nanmean(v)), 4),
                             std_cindex=round(float(np.nanstd(v)), 4),
                             n_fits=int(np.sum(~np.isnan(v))),
                             vae_params=n_params, vae_pretrain_sec=vae_secs,
                             survival_fit_sec=surv_secs, device=str(DEVICE)))
        print(f"  k={kk:>6}: VAE {vae_secs}s | survival {surv_secs}s | "
              + " ".join(f"{n} {np.nanmean(v):.3f}" for n, v in per_model.items()))
        del vae; gc.collect()
        if DEVICE.type == "cuda": torch.cuda.empty_cache()

    # Same config, forced onto CPU — the actual "can a student run this?" test.
    if cpu_check_k:
        cpu = torch.device("cpu")
        cols = top_variance_genes(cpu_check_k)
        vae, vae_secs, n_params = train_vae_dims(512, 32, cols=cols, epochs=min(20, SWEEP_VAE_EPOCHS),
                                                 device=cpu)
        cohort = cohorts[0]
        t0, cis = time.time(), []
        for seed in seeds[:3]:
            set_seed(seed)
            tr, te = split_indices(cohort, seed)
            Xtr, Xte, ytr, yte, ctr, cte = split_arrays(cohort, tr, te, cols=cols)
            _, risk = fit_risk("VAECox", Xtr, ytr, ctr, Xte, vae=vae, latent=32, device=cpu)
            cis.append(cindex_safe(yte, risk, cte))
        rows.append(dict(k_genes=cpu_check_k, model="VAECox (CPU-only)",
                         mean_cindex=round(float(np.nanmean(cis)), 4),
                         std_cindex=round(float(np.nanstd(cis)), 4),
                         n_fits=len(cis), vae_params=n_params,
                         vae_pretrain_sec=vae_secs,
                         survival_fit_sec=round(time.time() - t0, 1), device="cpu"))
        print(f"  CPU-only check (k={cpu_check_k}, {min(20, SWEEP_VAE_EPOCHS)} VAE epochs): "
              f"{vae_secs}s pretrain + {rows[-1]['survival_fit_sec']}s for 3 fits on {cohort}, "
              f"C-index {rows[-1]['mean_cindex']}")
        del vae; gc.collect()

    df = pd.DataFrame(rows)
    df.to_csv(f"{RES}/feature_subset.csv", index=False)
    return df


print("### 6i.a lightweight models across all cohorts")
LIGHT_CANCER = lightweight_by_cancer()
LIGHT_GAP = lightweight_disparity(LIGHT_CANCER)
print("\n### 6i.b feature budget + CPU feasibility")
FEATSUB = feature_subset_accessibility()

## 6j · Interpretability, full-cohort robustness, KM everywhere, paper comparison

Four things §6 only did partially:

* **Permutation importance** — a model-agnostic, SHAP-style attribution. VAECox's
  weights are uninterpretable per gene (the encoder mixes all of them), so
  `|Cox weight|` cannot rank genes for it; shuffling one gene's column and
  measuring the C-index drop can.
* **Robustness on all 10 cohorts**, not just STAD — and each model is trained
  once per (cohort, seed) then scored under every corruption level, instead of
  refitting per level.
* **Kaplan–Meier for every cohort**, with log-rank p-values collected in one table.
* **Cell-by-cell comparison against the paper's Table 1** (needs `paper_table1.csv`;
  see the note in the code — the paper's numbers are deliberately not hard-coded).

In [ ]:
# ---------------------------------------------------------------------------
# 6j · Interpretability, robustness across every cohort, KM everywhere,
#      and a direct cell-by-cell comparison against the paper's Table 1.
# ---------------------------------------------------------------------------

# --- 6j.1 permutation importance (model-agnostic, SHAP-style) ---------------
def permutation_importance(cohort, name="VAECox", n_candidates=300,
                           n_repeats=3, top_k=25, seed=0):
    """Drop in test C-index when a gene's values are shuffled across patients.

    Model-agnostic, so VAECox (whose weights say nothing about single genes,
    because the encoder mixes all of them) and CoxRidge are on equal footing.
    Permuting all ~20k genes is wasteful, so a cheap CoxRidge |weight| screen
    picks `n_candidates` genes and only those are permuted.
    """
    set_seed(seed)
    tr, te = split_indices(cohort, seed)
    Xtr, Xte, ytr, yte, ctr, cte = split_arrays(cohort, tr, te)

    screen = _fit(CoxLinear(Xtr.shape[1]), Xtr, ytr, ctr, 1e-4, 1e-3, CFG["SURV_EPOCHS"])
    w = screen.fc1.weight.detach().cpu().numpy().reshape(-1)
    cand = np.argsort(-np.abs(w))[:n_candidates]

    model, risk = fit_risk(name, Xtr, ytr, ctr, Xte)
    base = cindex_safe(yte, risk, cte)
    if np.isnan(base):
        return pd.DataFrame()

    rng = np.random.default_rng(seed + 3000)
    rows = []
    for j in cand:
        drops = []
        for _ in range(n_repeats):
            Xp = Xte.copy()
            Xp[:, j] = Xp[rng.permutation(len(Xp)), j]
            ci = cindex_safe(yte, _risk(model, Xp), cte)
            if not np.isnan(ci):
                drops.append(base - ci)
        if drops:
            rows.append(dict(cancer=cohort, model=name, gene=GENES[j],
                             base_cindex=round(float(base), 4),
                             mean_drop=round(float(np.mean(drops)), 5),
                             std_drop=round(float(np.std(drops)), 5)))
    df = pd.DataFrame(rows).sort_values("mean_drop", ascending=False).head(top_k)
    df.insert(2, "rank", range(1, len(df) + 1))
    return df


def run_importance(cohorts=None, models=("CoxRidge", "VAECox")):
    cohorts = cohorts or CFG["PAPER_10"][:3]
    out = [permutation_importance(c, m) for c in cohorts for m in models]
    out = [d for d in out if len(d)]
    df = pd.concat(out, ignore_index=True) if out else pd.DataFrame()
    if len(df):
        df.to_csv(f"{RES}/permutation_importance.csv", index=False)
        for (c, m), sub in df.groupby(["cancer", "model"]):
            genes = ", ".join(sub.head(5).gene)
            print(f"  {c:5s} {m:9s} top-5: {genes}")
    return df


# --- 6j.2 robustness on every cohort ----------------------------------------
def robustness_all(cohorts=None, seeds=None,
                   miss=(0.0, 0.1, 0.25, 0.5), sigmas=(0.0, 0.5, 1.0, 2.0)):
    """Corruption is applied at *inference* only, so each model is trained once
    per (cohort, seed) and then scored under every corruption level — the §6a
    version refit from scratch for every level, which was ~8x the compute for
    the same answer."""
    cohorts = cohorts or CFG["PAPER_10"]
    seeds = seeds if seeds is not None else CFG["SEEDS"][:5]
    rows = []
    for cohort in cohorts:
        acc = {}   # (experiment, level) -> {model: [cindex per seed]}
        for seed in seeds:
            set_seed(seed)
            tr, te = split_indices(cohort, seed)
            Xtr, Xte, ytr, yte, ctr, cte = split_arrays(cohort, tr, te)
            fitted = {m: fit_risk(m, Xtr, ytr, ctr, Xte)[0] for m in ("CoxRidge", "VAECox")}
            rng_m = np.random.default_rng(seed + 1000)
            rng_n = np.random.default_rng(seed + 2000)
            for frac in miss:
                Xc = Xte * (rng_m.random(Xte.shape) >= frac)
                for m, mod in fitted.items():
                    acc.setdefault(("missing", f"{int(frac*100)}%"), {}).setdefault(m, []).append(
                        cindex_safe(yte, _risk(mod, Xc.astype(np.float32)), cte))
            for sig in sigmas:
                Xc = (Xte + rng_n.normal(0, sig, Xte.shape)).astype(np.float32)
                for m, mod in fitted.items():
                    acc.setdefault(("noise", f"sigma={sig}"), {}).setdefault(m, []).append(
                        cindex_safe(yte, _risk(mod, Xc), cte))
        for (exp, lev), d in acc.items():
            rows.append(dict(cancer=cohort, experiment=exp, level=lev,
                             CoxRidge=round(float(np.nanmean(d["CoxRidge"])), 4),
                             VAECox=round(float(np.nanmean(d["VAECox"])), 4)))
        print(f"  {cohort} done")
    df = pd.DataFrame(rows)
    df.to_csv(f"{RES}/robustness_by_cancer.csv", index=False)
    # Headline: relative degradation from the clean baseline, averaged over cohorts.
    for exp in ("missing", "noise"):
        sub = df[df.experiment == exp]
        if not len(sub): continue
        piv = sub.pivot_table(index="level", values=["CoxRidge", "VAECox"], aggfunc="mean")
        clean = piv.iloc[0]
        rel = (piv - clean) / clean * 100
        print(f"\n  {exp}: % change in C-index vs clean (mean over cohorts)")
        print(rel.round(1).to_string())
    return df


# --- 6j.3 Kaplan-Meier for every cohort -------------------------------------
def kaplan_meier_all(cohorts=None, name="VAECox", seed=0):
    cohorts = cohorts or CFG["PAPER_10"]
    rows, panels = [], [c for c in cohorts if c in COHORT_DFS]
    ncol = min(5, max(1, len(panels)))
    nrow = int(np.ceil(len(panels) / ncol))
    fig, axes = plt.subplots(nrow, ncol, figsize=(3.2 * ncol, 3.0 * nrow), squeeze=False)
    for ax, cohort in zip(axes.ravel(), panels):
        set_seed(seed)
        tr, te = split_indices(cohort, seed)
        Xtr, Xte, ytr, yte, ctr, cte = split_arrays(cohort, tr, te)
        _, risk = fit_risk(name, Xtr, ytr, ctr, Xte)
        hi = risk >= np.median(risk)
        ev = (cte == 0)
        kmf = KaplanMeierFitter()
        for grp, lab in [(hi, "High risk"), (~hi, "Low risk")]:
            if grp.sum() > 0:
                kmf.fit(yte[grp], ev[grp], label=lab); kmf.plot_survival_function(ax=ax, ci_show=False)
        p = float("nan")
        if hi.sum() > 0 and (~hi).sum() > 0:
            p = logrank_test(yte[hi], yte[~hi], ev[hi], ev[~hi]).p_value
        ax.set_title(f"{cohort}  p={p:.3g}", fontsize=9)
        ax.set_xlabel("Days"); ax.set_ylabel("S(t)"); ax.legend(fontsize=7)
        rows.append(dict(cancer=cohort, model=name, n_test=int(len(yte)),
                         n_events=int(ev.sum()), n_high=int(hi.sum()), n_low=int((~hi).sum()),
                         log_rank_p=round(p, 5) if not np.isnan(p) else None,
                         significant=bool(p < 0.05) if not np.isnan(p) else None))
    for ax in axes.ravel()[len(panels):]:
        ax.axis("off")
    fig.tight_layout(); fig.savefig(f'{CFG["OUT"]}/figures/km_all_cohorts.png', dpi=120); plt.close(fig)
    df = pd.DataFrame(rows)
    df.to_csv(f"{RES}/km_summary.csv", index=False)
    n_sig = int(df.significant.fillna(False).sum())
    print(f"  risk stratification significant (p<0.05) in {n_sig}/{len(df)} cohorts")
    return df


# --- 6j.4 comparison against the paper's own Table 1 ------------------------
def compare_to_paper():
    """Merge our C-index table with the paper's Table 1, if it has been supplied.

    The paper's numbers are NOT hard-coded here — transcribe them yourself into
    `paper_table1.csv` (template in the repo: cancer,model,paper_cindex) and put
    it in the working dir or attach it as a Kaggle dataset. Without it this cell
    prints a reminder and skips, so nothing invented ends up in the manuscript.
    """
    hits = ([p for p in [f'{CFG["OUT"]}/paper_table1.csv', "paper_table1.csv",
                         "paper/paper_table1.csv"] if os.path.exists(p)] +
            glob.glob("/kaggle/input/**/paper_table1.csv", recursive=True))
    if not hits:
        print("  paper_table1.csv not found — skipping. Fill in the template from "
              "Table 1 of the paper to get a per-cell comparison.")
        return pd.DataFrame()
    paper = pd.read_csv(hits[0], comment="#", skip_blank_lines=True)
    paper.columns = [c.strip().lower() for c in paper.columns]
    paper = paper.dropna(subset=["paper_cindex"])
    merged = PH2_LONG.merge(paper, on=["cancer", "model"], how="inner")
    if not len(merged):
        print(f"  {hits[0]} has no rows matching our cancer/model names — skipping.")
        return pd.DataFrame()
    merged["delta"] = (merged.mean_cindex - merged.paper_cindex).round(4)
    merged["abs_delta"] = merged.delta.abs()
    merged.to_csv(f"{RES}/paper_comparison.csv", index=False)
    print(f"  matched {len(merged)} cells from {hits[0]}")
    print(f"  mean |delta| = {merged.abs_delta.mean():.4f}   "
          f"max |delta| = {merged.abs_delta.max():.4f}")
    if merged.model.nunique() > 1 and len(merged) > 2:
        r = np.corrcoef(merged.mean_cindex, merged.paper_cindex)[0, 1]
        print(f"  corr(ours, paper) across all cells = {r:.3f}")
    return merged


print("### 6j.1 permutation importance");       PERM_IMP = run_importance()
print("\n### 6j.2 robustness, every cohort");   ROB_ALL  = robustness_all()
print("\n### 6j.3 Kaplan-Meier, every cohort"); KM_ALL   = kaplan_meier_all()
print("\n### 6j.4 vs paper Table 1");           PAPER_CMP = compare_to_paper()

## 6k · Export numbers for the manuscript

Everything above is dumped to `results/manuscript_numbers.json`, which
`paper/build_manuscript.py` reads to fill the manuscript tables — so no number
is ever retyped by hand. Extension figures are written alongside it.

In [ ]:
# ---------------------------------------------------------------------------
# 6k · Export every number the manuscript needs, plus the extension figures.
# ---------------------------------------------------------------------------
def _safe(df, cols=None):
    if df is None or len(df) == 0:
        return []
    return df[cols].to_dict("records") if cols else df.to_dict("records")


NUMBERS = dict(
    data=dict(
        source="TCGA via GenoTEX / UCSC Xena (HiSeqV2_PANCAN)",
        n_cohorts_loaded=len(COHORT_DFS),
        cohorts=sorted(COHORT_DFS),
        evaluated=CFG["PAPER_10"],
        n_genes=NUM_FEATURES,
        per_cohort={c: dict(n=int(len(d)),
                            events=int((d.censored == 0).sum()),
                            censor_pct=round(100 * float((d.censored == 1).mean()), 1),
                            median_survival_days=float(np.median(d.survival)))
                    for c, d in COHORT_DFS.items()},
    ),
    setup=dict(device=str(DEVICE),
               gpu=torch.cuda.get_device_name(0) if DEVICE.type == "cuda" else "CPU",
               vae_epochs=CFG["VAE_EPOCHS"], vae_hidden=CFG["HIDDEN"], vae_latent=CFG["LATENT"],
               vae_lr=CFG["VAE_LR"], vae_wd=CFG["VAE_WD"], vae_batch=CFG["VAE_BATCH"],
               surv_epochs=CFG["SURV_EPOCHS"], seeds=CFG["SEEDS"],
               hp_grid=[list(h) for h in HP_GRID]),
    phase2=dict(table=_safe(PH2_LONG), wins=PH2_WINS,
                mean_cindex={m: (None if np.isnan(v) else round(float(v), 4))
                             for m, v in PH2_WIDE["Mean"].items()},
                best_mean_model=str(PH2_WIDE["Mean"].idxmax()),
                paper_wins="VAECox 7/10"),
    extensions=dict(
        robustness=_safe(ROB_ALL),
        subgroup=_safe(SUBGROUP),
        subgroup_gaps=_safe(GAPS),
        lightweight=_safe(LIGHT_CANCER),
        lightweight_disparity=_safe(LIGHT_GAP),
        feature_subset=_safe(FEATSUB),
        importance=_safe(PERM_IMP),
        kaplan_meier=_safe(KM_ALL),
    ),
    paper_comparison=_safe(PAPER_CMP),
)

with open(f"{RES}/manuscript_numbers.json", "w") as f:
    json.dump(NUMBERS, f, indent=2, default=str)
print(f"wrote {RES}/manuscript_numbers.json")


# ---- extension figures ------------------------------------------------------
FIG = f'{CFG["OUT"]}/figures'

# E1 · robustness curves, pooled over all cohorts
if len(ROB_ALL):
    pooled = ROB_ALL.groupby(["experiment", "level"], sort=False)[["CoxRidge", "VAECox"]].mean()
    fig, axes = plt.subplots(1, 2, figsize=(11, 4))
    for ax, exp in zip(axes, ["missing", "noise"]):
        sub = pooled.loc[exp] if exp in pooled.index.get_level_values(0) else None
        if sub is None: continue
        ax.plot(range(len(sub)), sub.CoxRidge, "-o", label="CoxRidge")
        ax.plot(range(len(sub)), sub.VAECox, "-o", label="VAECox")
        ax.set_xticks(range(len(sub))); ax.set_xticklabels(sub.index, rotation=20)
        ax.set_ylabel("C-index"); ax.legend()
        ax.set_title(f"Robustness — {exp} (mean over {ROB_ALL.cancer.nunique()} cohorts)")
    fig.tight_layout(); fig.savefig(f"{FIG}/ext_robustness_all.png", dpi=120); plt.close(fig)

# E2 · lightweight trade-off: params vs C-index, sized by train time
if len(LIGHT_CANCER):
    agg = LIGHT_CANCER.groupby("config").agg(
        n_params=("n_params", "first"), train_sec=("train_sec", "first"),
        mean_cindex=("cindex", "mean")).reset_index()
    fig, ax = plt.subplots(figsize=(6.5, 4.5))
    ax.scatter(agg.n_params / 1e6, agg.mean_cindex,
               s=30 + 4 * agg.train_sec, alpha=.7, color="#4C78A8")
    for _, r in agg.iterrows():
        ax.annotate(r.config, (r.n_params / 1e6, r.mean_cindex),
                    textcoords="offset points", xytext=(6, 4), fontsize=8)
    ax.set_xscale("log"); ax.set_xlabel("VAE parameters (millions, log scale)")
    ax.set_ylabel("Mean C-index"); ax.set_title("Lightweight VAEs — size vs accuracy (bubble = train seconds)")
    fig.tight_layout(); fig.savefig(f"{FIG}/ext_lightweight.png", dpi=120); plt.close(fig)

# E3 · subgroup gaps
if len(GAPS):
    top = GAPS.sort_values("gap", ascending=False).head(15)[::-1]
    fig, ax = plt.subplots(figsize=(7.5, 5))
    ax.barh([f"{r.cancer} · {r.variable} ({r.model})" for _, r in top.iterrows()],
            top.gap, color="#E45756")
    ax.set_xlabel("C-index gap (best group − worst group)")
    ax.set_title("Largest within-cohort subgroup disparities")
    fig.tight_layout(); fig.savefig(f"{FIG}/ext_subgroup_gaps.png", dpi=120); plt.close(fig)

# E4 · feature-subset accessibility
if len(FEATSUB):
    fig, ax = plt.subplots(figsize=(6.5, 4.5))
    for name, sub in FEATSUB.groupby("model"):
        sub = sub.sort_values("k_genes")
        ax.plot(sub.k_genes, sub.mean_cindex, "-o", label=name)
    ax.set_xscale("log"); ax.set_xlabel("Number of genes retained (log scale)")
    ax.set_ylabel("Mean C-index"); ax.legend()
    ax.set_title("Low-resource accessibility — C-index vs feature budget")
    fig.tight_layout(); fig.savefig(f"{FIG}/ext_feature_subset.png", dpi=120); plt.close(fig)

# E5 · lightweight disparity: does shrinking hurt small cohorts more?
if len(LIGHT_GAP):
    fig, ax = plt.subplots(figsize=(6.5, 4.5))
    for cfg_name, sub in LIGHT_GAP.groupby("config"):
        ax.scatter(sub.n_events, sub.delta_vs_full, label=cfg_name, alpha=.8)
    ax.axhline(0, color="grey", lw=1, ls="--")
    ax.set_xlabel("Uncensored events in cohort")
    ax.set_ylabel("Δ C-index vs full-size VAE")
    ax.legend(); ax.set_title("Do lightweight models hurt small cohorts more?")
    fig.tight_layout(); fig.savefig(f"{FIG}/ext_lightweight_disparity.png", dpi=120); plt.close(fig)

print("\nExtension figures written to", FIG)
for fn in sorted(os.listdir(FIG)):
    print("  ", fn)

## 7 · Figures & reproducibility card

In [ ]:
# Bar chart of mean C-index per model
fig, ax = plt.subplots(figsize=(7, 4))
PH2_WIDE["Mean"].sort_values().plot.barh(ax=ax, color="#4C78A8")
ax.set_xlabel("Mean C-index"); ax.set_title("VAECox reproduction — mean C-index across 10 cancers")
fig.tight_layout(); fig.savefig(f'{CFG["OUT"]}/figures/mean_cindex.png', dpi=120)

# Robustness figure
fig, ax = plt.subplots(figsize=(7, 4))
miss = ROB[ROB.experiment == "missing"]
ax.plot(range(len(miss)), miss.CoxRidge, "-o", label="CoxRidge")
ax.plot(range(len(miss)), miss.VAECox, "-o", label="VAECox")
ax.set_xticks(range(len(miss))); ax.set_xticklabels(miss.level)
ax.set_xlabel("Missing features"); ax.set_ylabel("C-index"); ax.legend()
ax.set_title("Robustness to missing features")
fig.tight_layout(); fig.savefig(f'{CFG["OUT"]}/figures/robustness.png', dpi=120)

card = f"""
================================================================================
REPRODUCIBILITY CARD — VAECox (Bioinformatics 2020, Suppl. 1)
================================================================================
Paper : Kim, Kim, Choe, Lee, Kang. "Improved survival analysis by learning
        shared genomic information from pan-cancer data."
        Bioinformatics 36(Suppl_1):i389-i398. DOI:10.1093/bioinformatics/btaa462
Claim : VAECox outperforms CoxLasso/CoxRidge/Coxnnet on 7/10 TCGA cancers (C-index).

DATA SOURCE     : REAL TCGA via GenoTEX (UCSC Xena TCGA Hub, HiSeqV2_PANCAN)
Cohorts         : {sorted(COHORT_DFS)}
Genes (VAE dim) : {NUM_FEATURES}
Device          : {DEVICE} ({torch.cuda.get_device_name(0) if DEVICE.type=='cuda' else 'CPU'})
Seeds           : {CFG['SEEDS']}

VAE             : {NUM_FEATURES}->{CFG['HIDDEN']}->{CFG['LATENT']} (mu,sigma), Tanh, Adam
                  lr={CFG['VAE_LR']} wd={CFG['VAE_WD']} epochs={CFG['VAE_EPOCHS']} batch={CFG['VAE_BATCH']}
VAECox          : pretrained encoder FINE-TUNED + Coxnnet(128)  [paper's method]
HP search       : {'reduced grid ' + str(HP_GRID) + ' per cancer' if CFG['HP_SEARCH'] else 'none'}
Surv epochs     : {CFG['SURV_EPOCHS']}

WINS (this run) : {PH2_WINS}
Paper wins      : VAECox 7/10

DEVIATIONS
  - HP search reduced vs paper's 18-combo 5-fold CV (time). Documented.
  - VAE minibatched on GPU (paper full-batch); numerically equivalent.
  - GenoTEX HiSeqV2_PANCAN gene set ({NUM_FEATURES}) may differ slightly from
    paper's 20,502 (pan-cancer-normalised vs per-cohort). Documented.
================================================================================
"""
with open(f'{CFG["OUT"]}/results/reproducibility_card.txt', "w") as f:
    f.write(card)
print(card)

print("\nAll outputs written to:", CFG["OUT"])
for root, _, files in os.walk(CFG["OUT"]):
    for fn in files:
        print("  ", os.path.join(root, fn))

## 8 · Next step: the manuscript

This notebook produces every result and figure. The ReScience C **paper** is a
separate LaTeX document that reports: the reproduced C-index table vs the
paper's Table 1, the win-count, the extension results above, and a candid
discussion of deviations. Download `/kaggle/working/results/*.csv` and the
figures, and hand them to the manuscript draft.